In [1]:
import os
import pandas as pd
from clickhouse_driver import Client

client = Client(host='clickhouse', port=9000)

DUMP_DIR = '/home/jovyan/work/dumps'
os.makedirs(DUMP_DIR, exist_ok=True)

result = client.execute('SELECT version()')
print(f'ClickHouse version: {result[0][0]}')

ClickHouse version: 23.8.16.16


/tmp/ipykernel_21252/310528312.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
result = client.execute('SELECT dt, currency, rate_to_rub, nominal FROM currency_rates ORDER BY dt, currency')
df = pd.DataFrame(result, columns=['dt', 'currency', 'rate_to_rub', 'nominal'])
df.to_csv(f'{DUMP_DIR}/currency_rates.csv', index=False)
print(f'currency_rates: {len(df):,} строк → {DUMP_DIR}/currency_rates.csv')

currency_rates: 15,017 строк → /home/jovyan/work/dumps/currency_rates.csv


In [3]:
result = client.execute('''
    SELECT
        transaction_id, account_id, account_level,
        daily_limit_rub, monthly_limit_rub,
        amount, amount_rub, account_balance, currency,
        transaction_type, merchant_category, category_name,
        risk_score, is_online,
        country_code, country_name, region, risk_level,
        timestamp, transaction_date, ingested_at
    FROM transactions_valid
    ORDER BY transaction_date, transaction_id
''')
columns = [
    'transaction_id', 'account_id', 'account_level',
    'daily_limit_rub', 'monthly_limit_rub',
    'amount', 'amount_rub', 'account_balance', 'currency',
    'transaction_type', 'merchant_category', 'category_name',
    'risk_score', 'is_online',
    'country_code', 'country_name', 'region', 'risk_level',
    'timestamp', 'transaction_date', 'ingested_at'
]
df = pd.DataFrame(result, columns=columns)
df.to_csv(f'{DUMP_DIR}/transactions_valid.csv', index=False)
print(f'transactions_valid: {len(df):,} строк → {DUMP_DIR}/transactions_valid.csv')

transactions_valid: 35,150 строк → /home/jovyan/work/dumps/transactions_valid.csv


In [4]:
result = client.execute('''
    SELECT
        transaction_id, account_id, account_level,
        amount, account_balance, currency,
        transaction_type, merchant_category,
        country_code, timestamp, transaction_date, ingested_at
    FROM transactions_invalid
    ORDER BY transaction_id
''')
columns = [
    'transaction_id', 'account_id', 'account_level',
    'amount', 'account_balance', 'currency',
    'transaction_type', 'merchant_category',
    'country_code', 'timestamp', 'transaction_date', 'ingested_at'
]
df = pd.DataFrame(result, columns=columns)
df.to_csv(f'{DUMP_DIR}/transactions_invalid.csv', index=False)
print(f'transactions_invalid: {len(df):,} строк → {DUMP_DIR}/transactions_invalid.csv')

transactions_invalid: 1,401 строк → /home/jovyan/work/dumps/transactions_invalid.csv
